In [1]:
import sys
import warnings
from config import Config
from pipeline import Pipeline

warnings.filterwarnings('ignore')

config = Config()

config.csv_path = 'dbbe_full.csv'
config.output_dir = './results'
config.use_scratch = False

config.subset_fraction = 0.2
config.poem_subset_fraction = 0.3

config.batch_size = 32

config.verse_similarity_thresholds = [0.5,0.6]
config.leiden_resolutions = [0.4, 0.5]
config.n_bootstraps = 3
config.n_gridsearch_iterations = 3

config.poem_similarity_thresholds = [0.3, 0.4,0.5]

pipeline = Pipeline(config)
results = pipeline.run()


computational resources

cpu:
  cores: 32
  usage: 62.5%

system memory:
  total: 376.0 gb
  used: 116.4 gb (32.7%)
  available: 253.3 gb

gpus: 1

  gpu 0: Tesla V100-SXM2-32GB
    memory total: 31.7 gb
    memory allocated: 0.0 gb (0.0%)
    memory reserved: 0.0 gb (0.0%)
    compute capability: 7.0


2025-11-27 15:14:36.624056: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
embedding: 100%|██████████| 424/424 [00:33<00:00, 12.70it/s]



pipeline completed
  performance report: ./performance_report.txt
  enriched dataset: ./enriched_dataset.csv

computational resources

cpu:
  cores: 32
  usage: 60.7%

system memory:
  total: 376.0 gb
  used: 118.4 gb (33.2%)
  available: 251.2 gb

gpus: 1

  gpu 0: Tesla V100-SXM2-32GB
    memory total: 31.7 gb
    memory allocated: 0.4 gb (1.1%)
    memory reserved: 1.5 gb (4.6%)
    compute capability: 7.0


In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv('enriched_dataset.csv')
df=df.dropna()
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    example_clusters = np.random.choice(clusters_with_multiple, size=min(n_clusters, len(clusters)), replace=False)
    
    for cid in example_clusters:
        print(f"Cluster {cid}")
        
        subset = df[df['predicted_verse_cluster'] == cid]
        
        for _, row in subset.iterrows():
            print(f"Verse: {row['verse']}")
        
        print("\n")
show_example_verse_clusters(df, n_clusters=10)

Cluster 563
Verse: πάντα τὲ ἀτρεκέως ἐξερέεινε βροτοῖς· |
Verse: πάντα τε ἀτρεκέως ἐξερέεινε βροτοῖς· |
Verse: πάντά τε ἀτρεκέως ἐξερέεινε βροτοῖς
Verse: Πάντα τε ἀτρεκέως ἐξερέεινε βροτοῖς,
Verse: πάντα τε ἀτρεκέ(ως), ἐξερέεινε βροτοῖς·
Verse: πάντα τε ἀτρεκέ(ως) ἐξερέεινε βροτοῖς·
Verse: πάντα τε ἀτρεκέως ἐξερέεινε βροτοῖς:
Verse: πάντ(α) τὲ ἀτρεκέ(ως) ἐξ ἐρέεινε βροτ(οῖς)·
Verse: πάντ(α) τὲ ἀτρεκέ(ως) ἐξερέεινε βροτ(οῖς)· |


Cluster 1818
Verse: δόξα τῶ λόγω τῶ δόντι τέλος· ἀμήν :
Verse: δόξα τῶ Θ(ε)ῶ πάντων ἕνεκα ἀμήν:-
Verse: + δόξα τῶ θ(ε)ῶ πάντων ἕνεκα· ἀμήν +
Verse: δόξα τῶ αγίω θ(ε)ω: ἀμήν: ξθϡπλ(...)


Cluster 739
Verse: εὖ συμπεράν(ας) σῆ χάριτι χ(ριστ)ε μου |
Verse: + ※ + χρηστ(ῶν) χορηγὲ χ(ριστ)ὲ χρηστά μοι δίδου ⁘
Verse: πάρασχε μοι τέλος τε χρηστὸν χ(ριστ)έ μου·
Verse: πολλὴ χάρις σοι παν|τοποιὲ χ(ριστ)έ μου :·
Verse: πολλὴ χάρις σοι παν|τοποιὲ χ(ριστ)έ μου :·
Verse: πάρασχε μοι τέλος τε χρηστὸν χ(ριστ)έ μου·


Cluster 6295
Verse: πρωτοφανῆ γετῆρα Θ(εὸ)ν αὐτογένεθλον.
Ve

In [3]:
import pandas as pd

poems_df = pd.read_csv("enriched_dataset.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

verse_dict = {}  
chunksize = 100_000

for chunk in pd.read_csv("dbbe_full.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)


Found 4643 multi-poem clusters.

CLUSTER 0

--- Poem 16897 ---

Εἰς νόον αἰγλήεντα θεόγραφα χείλεα βάψας,
κάλλεα ποικίλλεις ἱερώνυμα καὶ μετὰ πότμον·
ζωοσόφοις λογίοις κελαδῶν θεοφάντορας ὕμν(ους):-

----------------------------------------

--- Poem 17415 ---

+ εἰς νόον ἀιγλήεντα θεόγραφα | χείλεα βάψας·
κάλλεα ποικίλ|λεις ἱερώνυμα καὶ μετὰ πότ|μον· 
ζωοσόφοις λογίοις κελα|δὼν θεοφάντορας ὕμνους ⁘

----------------------------------------

--- Poem 17955 ---

⁘ εἰς νόον αἰγλήεντα θεόγραφα χείλεα βάψας, 
κάλλεα ποικί|λλεις ἱερώνυμα κ(αὶ) μετὰ πότμον, 
ζωοσόφοις λογί(οις) κελαδῶν θεο|φάντορ(ας) ὕμνους:-

----------------------------------------

--- Poem 18038 ---

Εἰς νόον αἰγλήεντα· θεόγραφα χεί|λεα βάψας,
καλλεὰ ποικί|λλεις ἱερώνυμα· καὶ μετὰ πότμον:· |
ζωοσόφοις λογίοις κελαδὼν, θε|οφάντορας ὕμνους:·

----------------------------------------

--- Poem 18027 ---

εἰς νόον αἰγλήεντα θεόγραφα χείλεα βάψας.
κάλλεα ποικίλλεις ἱερώνυμα κ(αὶ) μετὰ πότμον.
ζωοσόφοις λογίοις κελαδῶν θεοφάντ